# Amazon

## Diagnostics

In [ ]:
# ── Diagnostic Cell — Full flow: search → product → reviews ──────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, random

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

# ── Step 1: Search ─────────────────────────────────────────────────────────────
driver.get("https://www.amazon.com/")
time.sleep(3)
search = driver.find_element(By.ID, "twotabsearchtextbox")
for char in "wireless earbuds":
    search.send_keys(char)
    time.sleep(random.uniform(0.05, 0.15))
search.send_keys(Keys.ENTER)
time.sleep(4)

print("=== SEARCH RESULTS PAGE ===")
print("Title:", driver.title)
print("URL  :", driver.current_url)

# ── Step 2: Find and enter first product ───────────────────────────────────────
product_url = None
for sel in [
    "[data-component-type='s-search-result'] h2 a",
    ".s-result-item h2 a",
    "h2 a.a-link-normal",
    ".a-link-normal[href*='/dp/']",
]:
    els = driver.find_elements(By.CSS_SELECTOR, sel)
    if els:
        product_url = els[0].get_attribute("href")
        print(f"\n✅ Found product link via: '{sel}'")
        print(f"   URL: {product_url[:80]}")
        break
    else:
        print(f"❌ No match: '{sel}'")

if not product_url:
    print("\n🛑 Could not find product link — check browser for CAPTCHA")
    # Leave open for manual inspection
else:
    driver.get(product_url)
    time.sleep(4)
    if len(driver.window_handles) > 1:
        driver.switch_to.window(driver.window_handles[-1])

    print("\n=== PRODUCT PAGE ===")
    print("Title:", driver.title)
    print("URL  :", driver.current_url)

    # ── Step 3: Test product info selectors ───────────────────────────────────
    print("\n=== PRODUCT INFO SELECTORS ===")
    info_selectors = {
        "name":           ["#productTitle"],
        "brand":          ["#bylineInfo", "a#bylineInfo"],
        "price":          [".a-price .a-offscreen", "#corePriceDisplay_desktop_feature_div .a-offscreen",
                           "#price_inside_buybox", ".priceToPay .a-offscreen"],
        "rating_score":   ["#acrPopover .a-size-base.a-color-base", "span.a-icon-alt",
                           "[data-hook='rating-out-of-text']"],
        "rating_count":   ["#acrCustomerReviewText", "[data-hook='total-review-count']"],
        "availability":   ["#availability span"],
    }

    for field, sels in info_selectors.items():
        found = False
        for sel in sels:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els and els[0].text.strip():
                print(f"✅ {field:20s} via '{sel}' → '{els[0].text.strip()[:60]}'")
                found = True
                break
        if not found:
            print(f"❌ {field:20s} → no selector worked")

    # ── Step 4: Test reviews page selectors ───────────────────────────────────
    try:
        asin = driver.current_url.split("/dp/")[1].split("/")[0].split("?")[0]
    except:
        asin = driver.find_element(By.CSS_SELECTOR, "#ASIN").get_attribute("value")

    print(f"\n=== REVIEWS PAGE (ASIN: {asin}) ===")
    driver.get(f"https://www.amazon.com/product-reviews/{asin}/?sortBy=recent&pageNumber=1")
    time.sleep(4)

    print("Title:", driver.title)
    print("URL  :", driver.current_url)

    review_selectors = {
        "review card":   ["[data-hook='review']", ".review", "[id^='customer_review']"],
        "reviewer name": [".a-profile-name"],
        "star rating":   ["[data-hook='review-star-rating'] span", "[data-hook='cmps-review-star-rating'] span"],
        "review date":   ["[data-hook='review-date']"],
        "review title":  ["[data-hook='review-title'] span:not(.a-icon-alt)"],
        "review body":   ["[data-hook='review-body'] span"],
        "helpful votes": ["[data-hook='helpful-vote-statement']"],
        "next button":   ["li.a-last:not(.a-disabled) a", "li.a-last a", ".a-pagination .a-last a"],
    }

    print("\n=== REVIEW SELECTORS ===")
    for field, sels in review_selectors.items():
        found = False
        for sel in sels:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els:
                sample = els[0].text.strip()[:60] if els[0].text.strip() else "(no text)"
                print(f"✅ {field:20s} via '{sel}' → {len(els)} found | sample: '{sample}'")
                found = True
                break
        if not found:
            print(f"❌ {field:20s} → no selector worked")

    # ── Step 5: Dump first review card HTML ───────────────────────────────────
    print("\n=== FIRST REVIEW CARD HTML ===")
    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")
    if cards:
        print(cards[0].get_attribute("outerHTML")[:3000])
    else:
        print("No review cards found — dumping page body snippet:")
        print(driver.page_source[3000:6000])

print("\n✋ Browser left open — run driver.quit() when done")

## Diagnostics V2

In [ ]:
# ── Diagnostic Cell — H2 + SAME PARENT span extraction ────────────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, random, json

# ── Driver ────────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

# ── Helpers ───────────────────────────────────────────────────────────────────
def human_sleep(a=2, b=4):
    time.sleep(3)

def get_first_text(ctx, selectors):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val:
                print(f"   ✅ '{sel}' → {val[:60]}")
                return val
        except:
            print(f"   ❌ '{sel}'")
    return "N/A"

# ── FIXED DESCRIPTION (same parent) ───────────────────────────────────────────
def get_h2_description_same_parent(driver):
    print("\n=== H2 SAME-PARENT DESCRIPTION DEBUG ===")

    h2_elements = driver.find_elements(By.TAG_NAME, "h2")
    print(f"Found {len(h2_elements)} <h2> tags")

    for h2 in h2_elements:
        text = h2.text.strip()
        print(f"   🔎 H2: '{text}'")

        if "Product description" in text:
            print("   ✅ Matched target H2")

            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", h2)
            time.sleep(1)

            try:
                parent = h2.find_element(By.XPATH, "..")
                print(f"   📦 Parent tag: {parent.tag_name}")

                # get ALL spans inside same parent
                spans = parent.find_elements(By.TAG_NAME, "span")

                texts = [s.text.strip() for s in spans if s.text.strip()]

                if texts:
                    print(f"   ✅ Found {len(texts)} span(s) in same parent")
                    print(f"   📄 Preview: {texts[0][:120]}...")
                    return " ".join(texts)

                else:
                    print("   ⚠️ No text in spans under parent")

            except Exception as e:
                print(f"   ❌ Parent extraction failed: {e}")

    print("   ❌ No matching Product Description block found")
    return "N/A"

# ── Step 1: Search ───────────────────────────────────────────────────────────
driver.get("https://www.amazon.com/")
human_sleep()

search = driver.find_element(By.ID, "twotabsearchtextbox")
search.send_keys("wireless earbuds")
search.send_keys(Keys.ENTER)
human_sleep(3,5)

print("\n=== SEARCH PAGE ===")
print(driver.title)

# ── Step 2: First product ────────────────────────────────────────────────────
links = driver.find_elements(By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")
product_url = None

for el in links:
    href = el.get_attribute("href")
    if "/dp/" in href:
        product_url = href.split("?")[0]
        break

if not product_url:
    print("🛑 No product found")
else:
    print(f"\n➡️ Using product:\n{product_url}")
    driver.get(product_url)
    human_sleep(3,5)

    print("\n=== PRODUCT PAGE ===")
    print(driver.title)

    # ── Product fields ───────────────────────────────────────────────────────
    print("\n=== PRODUCT FIELD TEST ===")

    product = {
        "name": get_first_text(driver, ["#productTitle"]),
        "brand": get_first_text(driver, ["#bylineInfo"]),
        "price": get_first_text(driver, [".a-price .a-offscreen"]),
        "rating_score": get_first_text(driver, ["[data-hook='rating-out-of-text']"]),
        "rating_count": get_first_text(driver, ["#acrCustomerReviewText"]),
        "availability": get_first_text(driver, ["#availability span"])
    }

    # ── Description FIX ──────────────────────────────────────────────────────
    product["description"] = get_h2_description_same_parent(driver)

    # ── Output ───────────────────────────────────────────────────────────────
    print("\n=== FINAL OUTPUT SAMPLE ===")
    print(json.dumps(product, indent=2)[:1500])

print("\n✋ Done — driver still open")

In [ ]:
# ── Debug Cell — Find exact description selector ──────────────────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

# Go straight to the Crocs page we know has a description
driver.get("https://www.amazon.com/crocs-Unisex-Classic-Black-Women/dp/B0014BYHJE/?_encoding=UTF8&pd_rd_w=xznKY&content-id=amzn1.sym.a9c4acee-9ca0-46be-bae3-532a2b4b0d29%3Aamzn1.symc.5a16118f-86f0-44cd-8e3e-6c5f82df43d0&pf_rd_p=a9c4acee-9ca0-46be-bae3-532a2b4b0d29&pf_rd_r=RHRZG81JRK5XH6TNCQF6&pd_rd_wg=6CPLG&pd_rd_r=6c45d89b-724c-4395-977b-bbec99ff4500&ref_=pd_hp_d_atf_ci_mcx_mr_")
time.sleep(4)

# Scroll all the way down slowly to trigger lazy rendering
for _ in range(12):
    driver.execute_script("window.scrollBy(0, 500);")
    time.sleep(0.6)
time.sleep(2)

print("=== TESTING SELECTORS ===")
selectors = [
    "#productDescription p span",
    "#productDescription p",
    "#productDescription span",
    "#productDescription",
    "#productDescription_feature_div p span",
    "#productDescription_feature_div p",
    "#productDescription_feature_div span",
    "#productDescription_feature_div",
    "[data-feature-name='productDescription'] p span",
    "[data-feature-name='productDescription'] p",
    "[data-feature-name='productDescription'] span",
    "[data-template-name='productDescription'] p span",
    "[data-template-name='productDescription'] p",
]

for sel in selectors:
    els = driver.find_elements(By.CSS_SELECTOR, sel)
    if els:
        text = els[0].text.strip()
        print(f"✅ '{sel}'")
        print(f"   elements: {len(els)} | text: '{text[:80]}' | empty: {not bool(text)}")
    else:
        print(f"❌ '{sel}' → 0 elements")

# Also try via JavaScript in case Selenium can't see it
print("\n=== JAVASCRIPT INNERTEXT CHECK ===")
js_selectors = [
    "#productDescription p span",
    "#productDescription p",
    "#productDescription",
]
for sel in js_selectors:
    result = driver.execute_script(f"""
        var el = document.querySelector('{sel}');
        if (!el) return 'NOT FOUND';
        return el.innerText || el.textContent || 'EMPTY';
    """)
    print(f"JS '{sel}' → '{str(result)[:80]}'")

# Dump raw innerHTML of #productDescription so we can see exact structure
print("\n=== RAW innerHTML of #productDescription ===")
raw = driver.execute_script("var el = document.querySelector('#productDescription'); return el ? el.innerHTML : 'NOT FOUND';")
print(raw[:1500])

print("\n✋ Browser left open — run driver.quit() when done")
# ── Debug Cell — Find review body selector ────────────────────────────────────
# Run this AFTER the browser has a product page open with reviews visible
# Paste this in a new cell while driver is still alive

cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")
card = cards[0]  # inspect first card

print("=== RAW review card innerHTML (first 3000 chars) ===")
print(card.get_attribute("innerHTML")[:3000])

print("\n=== TESTING BODY SELECTORS ===")
body_selectors = [
    "span[data-hook='review-body']",
    "[data-hook='review-body'] span",
    "[data-hook='review-body'] p span",
    "[data-hook='review-body'] p",
    "[data-hook='review-body']",
    "div[data-hook='review-body']",
    "div[data-hook='review-body'] span",
    ".review-text",
    ".review-text-content",
    ".review-text-content span",
    "[class*='review-text'] span",
    "[class*='review-body'] span",
]

for sel in body_selectors:
    els = card.find_elements(By.CSS_SELECTOR, sel)
    if els:
        text = els[0].text.strip()
        print(f"✅ '{sel}' → {len(els)} found | text: '{text[:80]}' | empty: {not bool(text)}")
    else:
        print(f"❌ '{sel}' → 0 elements")

# ── Debug Cell — Find price selector ─────────────────────────────────────────
# Run while browser is open on a product page

print("=== TESTING PRICE SELECTORS ===")
price_selectors = [
    "#corePriceDisplay_desktop_feature_div .a-price-whole",
    ".a-price[data-a-color='price'] .a-offscreen",
    "#apex_offerDisplay_desktop .a-price .a-offscreen",
    "#corePrice_feature_div .a-price .a-offscreen",
    "#price .a-price .a-offscreen",
    "#priceblock_ourprice",
    "#priceblock_dealprice",
    ".a-price .a-offscreen",
    "#price_inside_buybox",
    ".priceToPay .a-offscreen",
    ".priceToPay span",
    "#apex_offerDisplay_desktop .a-price",
    "#newBuyBoxPrice",
    "[data-a-color='price'] .a-offscreen",
    "#buyNewSection .a-price .a-offscreen",
    "#corePrice_desktop_feature_div .a-price .a-offscreen",
    "#corePrice_desktop_feature_div .a-price-whole",
    "span.a-price.aok-align-center .a-offscreen",
    "#kindle-price",
    "#digital-list-price",
]

for sel in price_selectors:
    els = driver.find_elements(By.CSS_SELECTOR, sel)
    if els:
        text = els[0].text.strip()
        attr = els[0].get_attribute("innerHTML").strip()
        print(f"✅ '{sel}'")
        print(f"   .text='{text[:60]}' | innerHTML='{attr[:60]}'")
    else:
        print(f"❌ '{sel}'")

# Also try JS
print("\n=== JS PRICE CHECK ===")
result = driver.execute_script("""
    var selectors = [
        '.a-price .a-offscreen',
        '.priceToPay .a-offscreen',
        '#price_inside_buybox',
        '#corePrice_feature_div .a-offscreen',
        '#corePriceDisplay_desktop_feature_div .a-offscreen',
    ];
    for (var s of selectors) {
        var el = document.querySelector(s);
        if (el && el.innerText.trim()) return s + ' → ' + el.innerText.trim();
    }
    return 'none found';
""")
print("JS result:", result)

print("\n=== CURRENT URL ===")
print(driver.current_url)

# ── Debug Cell — Find price selector ─────────────────────────────────────────
# Run while browser is open on a product page

print("=== TESTING PRICE SELECTORS ===")
price_selectors = [
    "#corePriceDisplay_desktop_feature_div .a-price-whole",
    ".a-price[data-a-color='price'] .a-offscreen",
    "#apex_offerDisplay_desktop .a-price .a-offscreen",
    "#corePrice_feature_div .a-price .a-offscreen",
    "#price .a-price .a-offscreen",
    "#priceblock_ourprice",
    "#priceblock_dealprice",
    ".a-price .a-offscreen",
    "#price_inside_buybox",
    ".priceToPay .a-offscreen",
    ".priceToPay span",
    "#apex_offerDisplay_desktop .a-price",
    "#newBuyBoxPrice",
    "[data-a-color='price'] .a-offscreen",
    "#buyNewSection .a-price .a-offscreen",
    "#corePrice_desktop_feature_div .a-price .a-offscreen",
    "#corePrice_desktop_feature_div .a-price-whole",
    "span.a-price.aok-align-center .a-offscreen",
    "#kindle-price",
    "#digital-list-price",
]

for sel in price_selectors:
    els = driver.find_elements(By.CSS_SELECTOR, sel)
    if els:
        text = els[0].text.strip()
        attr = els[0].get_attribute("innerHTML").strip()
        print(f"✅ '{sel}'")
        print(f"   .text='{text[:60]}' | innerHTML='{attr[:60]}'")
    else:
        print(f"❌ '{sel}'")

# Also try JS
print("\n=== JS PRICE CHECK ===")
result = driver.execute_script("""
    var selectors = [
        '.a-price .a-offscreen',
        '.priceToPay .a-offscreen',
        '#price_inside_buybox',
        '#corePrice_feature_div .a-offscreen',
        '#corePriceDisplay_desktop_feature_div .a-offscreen',
    ];
    for (var s of selectors) {
        var el = document.querySelector(s);
        if (el && el.innerText.trim()) return s + ' → ' + el.innerText.trim();
    }
    return 'none found';
""")
print("JS result:", result)

print("\n=== CURRENT URL ===")
print(driver.current_url)

## Scraper V1

In [ ]:
# ── Single Cell — Amazon Product + Reviews Scraper (On-Page Reviews) ──────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from IPython.display import display
import pandas as pd
import time, json, random

# ── 1. Driver ─────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

def get_text(ctx, selector, default="N/A"):
    try: return ctx.find_element(By.CSS_SELECTOR, selector).text.strip()
    except: return default

def get_first_text(ctx, selectors, default="N/A"):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val: return val
        except: continue
    return default

def human_sleep(a=2, b=4):
    time.sleep(random.uniform(a, b))

def check_for_captcha(driver):
    if "captcha" in driver.page_source.lower() or "Type the characters" in driver.page_source:
        print("⚠️  CAPTCHA detected! Solve it in the browser then press Enter...")
        input()

try:
    # ── 2. Search ─────────────────────────────────────────────────────────────
    driver.get("https://www.amazon.com/")
    human_sleep(2, 3)
    check_for_captcha(driver)

    wait.until(EC.presence_of_element_located((By.ID, "twotabsearchtextbox")))
    search = driver.find_element(By.ID, "twotabsearchtextbox")
    search.clear()
    for char in "wireless earbuds":
        search.send_keys(char)
        time.sleep(random.uniform(0.05, 0.15))
    search.send_keys(Keys.ENTER)
    human_sleep(3, 5)
    check_for_captcha(driver)

    # ── 3. Open first product ─────────────────────────────────────────────────
    first = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")))
    product_url = first.get_attribute("href")
    driver.get(product_url)
    human_sleep(3, 5)
    check_for_captcha(driver)
    print(f"📦 {driver.title[:80]}")

    # ── 4. Scroll slowly to load all lazy content ─────────────────────────────
    for _ in range(8):
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(0.7)
    driver.execute_script("window.scrollTo(0, 0);")  # back to top
    human_sleep(1, 2)

    # ── 5. Product Info ───────────────────────────────────────────────────────
    product = {
        "url":          driver.current_url,
        "name":         get_text(driver, "#productTitle"),
        "brand":        get_text(driver, "#bylineInfo"),
        "rating_score": get_text(driver, "[data-hook='rating-out-of-text']"),
        "rating_count": get_text(driver, "#acrCustomerReviewText"),
        "price": get_first_text(driver, [
            "#corePriceDisplay_desktop_feature_div .a-price-whole",
            ".a-price[data-a-color='price'] .a-offscreen",
            "#apex_offerDisplay_desktop .a-price .a-offscreen",
            "#corePrice_feature_div .a-price .a-offscreen",
            "#price .a-price .a-offscreen",
        ]),
        "availability": get_first_text(driver, [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div span",
            "#deliveryBlockMessage span",
        ]),
        "answered_questions": get_text(driver, "#askATFLink span"),
    }

    # Highlights
    try:
        bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
        product["highlights"] = [b.text.strip() for b in bullets if b.text.strip()]
    except:
        product["highlights"] = []

    # Variants
    try:
        variant_groups = driver.find_elements(By.CSS_SELECTOR, ".a-section.a-spacing-small.poi-rotate-anchor")
        variants = {}
        for grp in variant_groups:
            label = get_text(grp, "label.a-form-label")
            opts  = [o.get_attribute("title") or o.text for o in grp.find_elements(By.CSS_SELECTOR, "li")]
            opts  = [o.strip() for o in opts if o and o.strip()]
            if label and opts:
                variants[label.rstrip(":")] = opts
        product["variants"] = variants
    except:
        product["variants"] = {}

    # Images
    try:
        thumbs = driver.find_elements(By.CSS_SELECTOR, "#altImages img")
        product["images"] = list({img.get_attribute("src") for img in thumbs if img.get_attribute("src")})
    except:
        product["images"] = []

    # Description
    try:
        desc_el = driver.find_element(By.CSS_SELECTOR, "#productDescription p, #productDescription_feature_div")
        product["description"] = desc_el.text.strip()
    except:
        product["description"] = "N/A"

    # Specs
    try:
        rows  = driver.find_elements(By.CSS_SELECTOR,
            "#productDetails_techSpec_section_1 tr, #productDetails_detailBullets_sections1 tr")
        specs = {}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                specs[cols[0].text.strip()] = cols[1].text.strip()
        if not specs:
            for item in driver.find_elements(By.CSS_SELECTOR, "#detailBullets_feature_div li"):
                text = item.text.strip()
                if ":" in text:
                    k, v = text.split(":", 1)
                    specs[k.strip().strip("\u200f")] = v.strip()
        product["specifications"] = specs
    except:
        product["specifications"] = {}

    print(f"✅ Name    : {product['name'][:70]}")
    print(f"   Price  : {product['price']}")
    print(f"   Rating : {product['rating_score']} — {product['rating_count']}")

    # ── 6. Scrape reviews directly on the product page ────────────────────────
    # Amazon shows ~8 reviews in the "Top reviews" section without login
    all_reviews = []

    try:
        review_section = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "[data-hook='review'], #cm-cr-dp-review-list")
        ))
    except:
        print("⚠️  Review section not found, scrolling more...")
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(0.8)

    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")
    print(f"\n📝 Found {len(cards)} reviews on product page")

    for card in cards:
        def g(sel, default="N/A"):
            try: return card.find_element(By.CSS_SELECTOR, sel).text.strip()
            except: return default

        # Star rating
        try:
            star_el = card.find_element(By.CSS_SELECTOR,
                "[data-hook='review-star-rating'] span, [data-hook='cmps-review-star-rating'] span")
            rating = star_el.get_attribute("innerHTML").split(" ")[0]
        except:
            rating = "N/A"

        # Review images
        try:
            imgs   = card.find_elements(By.CSS_SELECTOR, "[data-hook='review-image-tile']")
            images = [i.get_attribute("src") for i in imgs if i.get_attribute("src")]
        except:
            images = []

        all_reviews.append({
            "reviewer":      g(".a-profile-name"),
            "rating":        rating,
            "date":          g("[data-hook='review-date']"),
            "verified":      g("[data-hook='avp-badge']"),
            "title":         g("h5[data-hook='reviewTitle']"),
            "body":          g("p span, [data-hook='review-body'] p span, [data-hook='review-body'] p"),
            "helpful_votes": g("[data-hook='helpful-vote-statement']"),
            "variant":       g("[data-hook='format-strip']"),
            "image_count":   len(images),
            "images":        images,
        })

    print(f"🎉 Scraped {len(all_reviews)} reviews")

    # ── 7. Save ───────────────────────────────────────────────────────────────
    final = {**product, "reviews": all_reviews}
    with open("amazon_product.json", "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)

    df = pd.DataFrame(all_reviews).drop(columns=["images"], errors="ignore")
    df.to_csv("amazon_reviews.csv", index=False)
    print("💾 Saved: amazon_product.json  |  amazon_reviews.csv")
    display(df)

except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()
    print(f"\n🔍 URL  : {driver.current_url}")
    print(f"🔍 Title: {driver.title}")

finally:
    time.sleep(2)
    driver.quit()
    print("🛑 Driver closed")

## Scraper V2

In [ ]:
# ── Single Cell — Amazon Search Results + Per-Product Reviews Scraper ─────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from IPython.display import display
import pandas as pd
import time, json, random

# ── 1. Driver ─────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

def get_text(ctx, selector, default="N/A"):
    try: return ctx.find_element(By.CSS_SELECTOR, selector).text.strip()
    except: return default

def get_first_text(ctx, selectors, default="N/A"):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val: return val
        except: continue
    return default

def human_sleep(a=2, b=4):
    time.sleep(random.uniform(a, b))

def check_for_captcha(driver):
    if "captcha" in driver.page_source.lower() or "Type the characters" in driver.page_source:
        print("⚠️  CAPTCHA detected! Solve it in the browser then press Enter...")
        input()

def scrape_product(driver, url, index):
    """Navigate to a product page and scrape all info + on-page reviews."""
    driver.get(url)
    human_sleep(2, 4)
    check_for_captcha(driver)

    # ── Scroll slowly all the way down (12 steps) to trigger lazy rendering ───
    for _ in range(12):
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(0.6)
    time.sleep(2)  # extra wait after full scroll — this is what fixed description
    driver.execute_script("window.scrollTo(0, 0);")
    human_sleep(1, 2)

    # ── Description — read via JS to bypass any Selenium rendering lag ─────────
    description = driver.execute_script("""
        var el = document.querySelector('#productDescription p span');
        if (!el) el = document.querySelector('#productDescription p');
        if (!el) el = document.querySelector('#productDescription');
        return el ? (el.innerText || el.textContent || '').trim() : 'N/A';
    """) or "N/A"

    product = {
        "index":        index,
        "url":          driver.current_url,
        "name":         get_text(driver, "#productTitle"),
        "description":  description,
        "brand":        get_text(driver, "#bylineInfo"),
        "rating_score": get_text(driver, "[data-hook='rating-out-of-text']"),
        "rating_count": get_text(driver, "#acrCustomerReviewText"),
        "price": get_first_text(driver, [
            "#corePriceDisplay_desktop_feature_div .a-price-whole",
            ".a-price[data-a-color='price'] .a-offscreen",
            "#apex_offerDisplay_desktop .a-price .a-offscreen",
            "#corePrice_feature_div .a-price .a-offscreen",
            "#price .a-price .a-offscreen",
        ]),
        "availability": get_first_text(driver, [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div span",
            "#deliveryBlockMessage span",
        ]),
        "answered_questions": get_text(driver, "#askATFLink span"),
    }
    
    product_check(product)

    # Highlights
    try:
        bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
        product["highlights"] = [b.text.strip() for b in bullets if b.text.strip()]
    except:
        product["highlights"] = []

    # Variants
    try:
        variant_groups = driver.find_elements(By.CSS_SELECTOR, ".a-section.a-spacing-small.poi-rotate-anchor")
        variants = {}
        for grp in variant_groups:
            label = get_text(grp, "label.a-form-label")
            opts  = [o.get_attribute("title") or o.text for o in grp.find_elements(By.CSS_SELECTOR, "li")]
            opts  = [o.strip() for o in opts if o and o.strip()]
            if label and opts:
                variants[label.rstrip(":")] = opts
        product["variants"] = variants
    except:
        product["variants"] = {}

    # Images
    try:
        thumbs = driver.find_elements(By.CSS_SELECTOR, "#altImages img")
        product["images"] = list({img.get_attribute("src") for img in thumbs if img.get_attribute("src")})
    except:
        product["images"] = []

    # Description
    try:
        desc_el = driver.find_element(By.CSS_SELECTOR, "#productDescription p span")
        product["description"] = desc_el.text.strip()
    except:
        product["description"] = "N/A"

    # Specs
    try:
        rows  = driver.find_elements(By.CSS_SELECTOR,
            "#productDetails_techSpec_section_1 tr, #productDetails_detailBullets_sections1 tr")
        specs = {}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                specs[cols[0].text.strip()] = cols[1].text.strip()
        if not specs:
            for item in driver.find_elements(By.CSS_SELECTOR, "#detailBullets_feature_div li"):
                text = item.text.strip()
                if ":" in text:
                    k, v = text.split(":", 1)
                    specs[k.strip().strip("\u200f")] = v.strip()
        product["specifications"] = specs
    except:
        product["specifications"] = {}

    # ── Reviews (on-page only, no navigation) ─────────────────────────────────
    reviews = []
    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")

    for card in cards:
        def g(sel, default="N/A"):
            try: return card.find_element(By.CSS_SELECTOR, sel).text.strip()
            except: return default

        try:
            star_el = card.find_element(By.CSS_SELECTOR,
                "[data-hook='review-star-rating'] span, [data-hook='cmps-review-star-rating'] span")
            rating = star_el.get_attribute("innerHTML").split(" ")[0]
        except:
            rating = "N/A"

        try:
            imgs   = card.find_elements(By.CSS_SELECTOR, "[data-hook='review-image-tile']")
            images = [i.get_attribute("src") for i in imgs if i.get_attribute("src")]
        except:
            images = []

        reviews.append({
            "reviewer":      g(".a-profile-name"),
            "rating":        rating,
            "date":          g("[data-hook='review-date']"),
            "verified":      g("[data-hook='avp-badge']"),
            "title":         g("h5[data-hook='reviewTitle']"),
            "body":          g("p span, [data-hook='review-body'] p span, [data-hook='review-body'] p"),
            "helpful_votes": g("[data-hook='helpful-vote-statement']"),
            "variant":       g("[data-hook='format-strip']"),
            "image_count":   len(images),
            "images":        images,
        })

    product["reviews"] = reviews
    field_check(reviews)
    return product

def product_check(product):
    # ── Live product field check ──────────────────────────────────────────────────
    REQUIRED_PRODUCT_FIELDS = ["name", "description", "brand", "price", "rating_score", "rating_count", "availability"]

    print(f"\n  📋 Field Check:")
    for field in REQUIRED_PRODUCT_FIELDS:
        val = product.get(field, "N/A")
        status = "✅" if val and val != "N/A" else "❌"
        preview = str(val)[:60] if val and val != "N/A" else "N/A"
        print(f"     {status} {field:20s}: {preview}")

def field_check(reviews):
    # ── Live field check — prints immediately after each review ───────────────
    last = reviews[-1]
    problems = [field for field, val in last.items()
                if val in ("N/A", "", None) and field not in ("images", "helpful_votes", "variant")]
    if problems:
        print(f"     ⚠️  Review #{len(reviews)} — N/A fields: {problems}")
        print(f"          title='{last['title']}' | body='{str(last['body'])[:40]}'")

        product["reviews"] = reviews
        return product
    
    print(f"     ✅ {product['name'][:55]}")
    print(f"     💬 {len(product['reviews'])} reviews | 💰 {product['price']} | ⭐ {product['rating_score']}")

    # ── Product-level N/A check ───────────────────────────────────────────────
    na_fields = [k for k, v in product.items()
                if v in ("N/A", "", None) and k not in ("images", "variants", "highlights", "reviews", "specifications", "description")]
    if na_fields:
        print(f"     ⚠️  Product N/A fields: {na_fields}")

try:
    # ── 2. Search ─────────────────────────────────────────────────────────────
    driver.get("https://www.amazon.com/")
    human_sleep(2, 3)
    check_for_captcha(driver)

    wait.until(EC.presence_of_element_located((By.ID, "twotabsearchtextbox")))
    search = driver.find_element(By.ID, "twotabsearchtextbox")
    search.clear()
    for char in "wireless earbuds":
        search.send_keys(char)
        time.sleep(random.uniform(0.05, 0.15))
    search.send_keys(Keys.ENTER)
    human_sleep(3, 5)
    check_for_captcha(driver)

    # ── 3. Collect all product URLs from search results page ──────────────────
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")))

    # Scroll results page to load all cards
    for _ in range(5):
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(0.5)

    # Grab unique /dp/ links only (filters out nav/ad junk)
    all_links = driver.find_elements(By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")
    product_urls = []
    seen = set()
    for el in all_links:
        href = el.get_attribute("href") or ""
        # Extract clean base URL up to /dp/ASIN
        if "/dp/" in href:
            try:
                asin  = href.split("/dp/")[1].split("/")[0].split("?")[0]
                clean = f"https://www.amazon.com/dp/{asin}"
                if asin not in seen and len(asin) == 10:  # ASINs are always 10 chars
                    seen.add(asin)
                    product_urls.append(clean)
            except:
                continue

    print(f"🔍 Found {len(product_urls)} unique products on results page")
    for i, u in enumerate(product_urls):
        print(f"   {i+1}. {u}")

    # ── 4. Scrape each product ─────────────────────────────────────────────────
    search_results_url = driver.current_url  # save to go back if needed
    all_products = []

    for i, url in enumerate(product_urls):
        print(f"\n{'─'*60}")
        print(f"📦 [{i+1}/{len(product_urls)}] Scraping: {url}")
        try:
            product = scrape_product(driver, url, i + 1)
            all_products.append(product)
            print(f"   ✅ {product['name'][:60]}")
            print(f"   💬 {len(product['reviews'])} reviews scraped")
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            all_products.append({"index": i+1, "url": url, "error": str(e), "reviews": []})

        human_sleep(2, 4)  # be polite between requests

    # ── 5. Save ───────────────────────────────────────────────────────────────
    with open("amazon_product.json", "w", encoding="utf-8") as f:
        json.dump(all_products, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Saved: amazon_product.json")

    # Flat reviews CSV — each row = one review, with product name column
    all_reviews_flat = []
    for p in all_products:
        for r in p.get("reviews", []):
            all_reviews_flat.append({
                "product_index": p.get("index"),
                "product_name":  p.get("name", "N/A"),
                "product_url":   p.get("url", "N/A"),
                **{k: v for k, v in r.items() if k != "images"},
            })

    df = pd.DataFrame(all_reviews_flat)
    df.to_csv("amazon_reviews.csv", index=False)
    print(f"💾 Saved: amazon_reviews.csv ({len(df)} total reviews)")

    # Summary table
    summary = pd.DataFrame([{
        "index":        p.get("index"),
        "name":         p.get("name", "N/A")[:50],
        "price":        p.get("price", "N/A"),
        "rating_score": p.get("rating_score", "N/A"),
        "rating_count": p.get("rating_count", "N/A"),
        "reviews_scraped": len(p.get("reviews", [])),
    } for p in all_products])

    print(f"\n📊 Summary ({len(all_products)} products):")
    display(summary)
    display(df.head(10))

except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()
    print(f"\n🔍 URL  : {driver.current_url}")
    print(f"🔍 Title: {driver.title}")

finally:
    time.sleep(2)
    driver.quit()
    print("🛑 Driver closed")

## Scraper V3

In [ ]:
# ── Single Cell — Amazon Best Sellers Scraper (Multi-Department) ──────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from IPython.display import display
import pandas as pd
import time, json, random

# ── 1. Departments ────────────────────────────────────────────────────────────
DEPARTMENTS = {
    "Electronics":               "https://www.amazon.com/Best-Sellers-Electronics/zgbs/electronics/",
    "Clothing, Shoes & Jewelry": "https://www.amazon.com/Best-Sellers-Clothing-Shoes-Jewelry/zgbs/fashion/",
    "Kitchen & Home":            "https://www.amazon.com/Best-Sellers-Kitchen-Dining/zgbs/kitchen/",
}

# ── 2. Driver ─────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

# ── 3. Helpers ────────────────────────────────────────────────────────────────
def get_text(ctx, selector, default="N/A"):
    try: return ctx.find_element(By.CSS_SELECTOR, selector).text.strip()
    except: return default

def get_first_text(ctx, selectors, default="N/A"):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val: return val
        except: continue
    return default

def human_sleep(a=2, b=4):
    time.sleep(random.uniform(a, b))

def check_for_captcha(driver):
    if "captcha" in driver.page_source.lower() or "Type the characters" in driver.page_source:
        print("⚠️  CAPTCHA detected! Solve it in the browser then press Enter...")
        input()

def product_check(product):
    REQUIRED = ["name", "description", "brand", "price", "rating_score", "rating_count", "availability"]
    print(f"\n  📋 Field Check:")
    for field in REQUIRED:
        val = product.get(field, "N/A")
        status = "✅" if val and val != "N/A" else "❌"
        print(f"     {status} {field:20s}: {str(val)[:60] if status == '✅' else 'N/A'}")

def review_check(reviews):
    if not reviews: return
    last = reviews[-1]
    problems = [f for f, v in last.items()
                if v in ("N/A", "", None) and f not in ("images", "helpful_votes", "variant")]
    if problems:
        print(f"     ⚠️  Review #{len(reviews)} N/A: {problems} | title='{last['title']}' | body='{str(last['body'])[:40]}'")

def get_product_urls(driver):
    for _ in range(10):          # more scrolls to reach all 50 items
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(0.8)          # longer wait between scrolls for lazy loading
    time.sleep(2)                # extra wait at the end
    links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/dp/']")
    urls, seen = [], set()
    for el in links:
        href = el.get_attribute("href") or ""
        if "/dp/" in href:
            try:
                asin = href.split("/dp/")[1].split("/")[0].split("?")[0]
                if asin not in seen and len(asin) == 10:
                    seen.add(asin)
                    urls.append(f"https://www.amazon.com/dp/{asin}")
            except: continue
    return urls[:50]             # cap at 50

def scrape_product(driver, url, index, department):
    driver.get(url)
    human_sleep(2, 4)
    check_for_captcha(driver)

    # Scroll slowly to trigger all lazy-loaded content
    for _ in range(12):
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(0.6)
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, 0);")
    human_sleep(1, 2)

    # ── Description via JS ────────────────────────────────────────────────────
    description = driver.execute_script("""
        var el = document.querySelector('#productDescription p span');
        if (!el) el = document.querySelector('#productDescription p');
        if (!el) el = document.querySelector('#productDescription');
        return el ? (el.innerText || el.textContent || '').trim() : null;
    """) or None

    # Fallback — "About this item" bullets if description is empty
    if not description:
        try:
            bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
            bullet_texts = [b.text.strip() for b in bullets if b.text.strip()]
            description = " | ".join(bullet_texts) if bullet_texts else "N/A"
        except:
            description = "N/A"

    # ── Price via JS (handles visually hidden .a-offscreen spans) ─────────────
    price = driver.execute_script("""
        var selectors = [
            '#corePrice_feature_div .a-offscreen',
            '#corePriceDisplay_desktop_feature_div .a-price-whole',
            '#apex_offerDisplay_desktop .a-price .a-offscreen',
            '.a-price .a-offscreen',
        ];
        for (var s of selectors) {
            var el = document.querySelector(s);
            if (el) {
                var val = (el.innerText || el.innerHTML || '').trim();
                if (val) return val;
            }
        }
        return 'N/A';
    """) or "N/A"

    # ── Product info ──────────────────────────────────────────────────────────
    product = {
        "department":         department,
        "index":              index,
        "url":                driver.current_url,
        "name":               get_text(driver, "#productTitle"),
        "description":        description,
        "brand":              get_text(driver, "#bylineInfo"),
        "rating_score":       get_text(driver, "[data-hook='rating-out-of-text']"),
        "rating_count":       get_text(driver, "#acrCustomerReviewText"),
        "price":              price,
        "availability": get_first_text(driver, [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div span",
            "#deliveryBlockMessage span",
        ]),
        "answered_questions": get_text(driver, "#askATFLink span"),
    }

    product_check(product)

    # ── Highlights ────────────────────────────────────────────────────────────
    try:
        bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
        product["highlights"] = [b.text.strip() for b in bullets if b.text.strip()]
    except:
        product["highlights"] = []

    # ── Variants ──────────────────────────────────────────────────────────────
    try:
        variant_groups = driver.find_elements(By.CSS_SELECTOR, ".a-section.a-spacing-small.poi-rotate-anchor")
        variants = {}
        for grp in variant_groups:
            label = get_text(grp, "label.a-form-label")
            opts  = [o.get_attribute("title") or o.text for o in grp.find_elements(By.CSS_SELECTOR, "li")]
            opts  = [o.strip() for o in opts if o and o.strip()]
            if label and opts:
                variants[label.rstrip(":")] = opts
        product["variants"] = variants
    except:
        product["variants"] = {}

    # ── Images ────────────────────────────────────────────────────────────────
    try:
        thumbs = driver.find_elements(By.CSS_SELECTOR, "#altImages img")
        product["images"] = list({img.get_attribute("src") for img in thumbs if img.get_attribute("src")})
    except:
        product["images"] = []

    # ── Specs ─────────────────────────────────────────────────────────────────
    try:
        rows = driver.find_elements(By.CSS_SELECTOR,
            "#productDetails_techSpec_section_1 tr, #productDetails_detailBullets_sections1 tr")
        specs = {}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                specs[cols[0].text.strip()] = cols[1].text.strip()
        if not specs:
            for item in driver.find_elements(By.CSS_SELECTOR, "#detailBullets_feature_div li"):
                text = item.text.strip()
                if ":" in text:
                    k, v = text.split(":", 1)
                    specs[k.strip().strip("\u200f")] = v.strip()
        product["specifications"] = specs
    except:
        product["specifications"] = {}

    # ── Reviews (on-page only) ────────────────────────────────────────────────
    reviews = []
    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")

    for card in cards:
        def g(sel, default="N/A"):
            for s in sel.split(" || "):
                try:
                    val = card.find_element(By.CSS_SELECTOR, s.strip()).text.strip()
                    if val: return val
                except: continue
            return default

        try:
            star_el = card.find_element(By.CSS_SELECTOR,
                "[data-hook='review-star-rating'] span, [data-hook='cmps-review-star-rating'] span")
            rating = star_el.get_attribute("innerHTML").split(" ")[0]
        except:
            rating = "N/A"

        try:
            imgs   = card.find_elements(By.CSS_SELECTOR, "[data-hook='review-image-tile']")
            images = [i.get_attribute("src") for i in imgs if i.get_attribute("src")]
        except:
            images = []

        try:
            body_els = card.find_elements(By.CSS_SELECTOR, "[class*='review-text'] span")
            body = " ".join([el.text.strip() for el in body_els if el.text.strip()]) or "N/A"
        except:
            body = "N/A"

        reviews.append({
            "reviewer":      g(".a-profile-name"),
            "rating":        rating,
            "date":          g("[data-hook='review-date']"),
            "verified":      g("[data-hook='avp-badge']"),
            "title":         g("a[data-hook='review-title'] h5 || h5[data-hook='reviewTitle'] || [data-hook='review-title']"),
            "body":          body,
            "helpful_votes": g("[data-hook='helpful-vote-statement']"),
            "variant":       g("[data-hook='format-strip']"),
            "image_count":   len(images),
            "images":        images,
        })

        review_check(reviews)

    product["reviews"] = reviews
    return product

# ── 4. Main loop ──────────────────────────────────────────────────────────────
all_products     = []
all_reviews_flat = []

try:
    for dept_name, dept_url in DEPARTMENTS.items():
        print(f"\n{'═'*60}")
        print(f"🏬 Department: {dept_name}")
        print(f"{'═'*60}")

        driver.get(dept_url)
        human_sleep(3, 5)
        check_for_captcha(driver)

        product_urls = get_product_urls(driver)
        print(f"🔍 Found {len(product_urls)} products (capped at 50)")

        for i, url in enumerate(product_urls):
            print(f"\n{'─'*60}")
            print(f"📦 [{i+1}/{len(product_urls)}] {url}")
            try:
                product = scrape_product(driver, url, i + 1, dept_name)
                all_products.append(product)
                print(f"   ✅ {product['name'][:60]}")
                print(f"   💬 {len(product['reviews'])} reviews")

                for r in product.get("reviews", []):
                    all_reviews_flat.append({
                        "department":    dept_name,
                        "product_index": i + 1,
                        "product_name":  product.get("name", "N/A"),
                        "product_url":   url,
                        **{k: v for k, v in r.items() if k != "images"},
                    })
            except Exception as e:
                print(f"   ❌ Failed: {e}")
                all_products.append({
                    "department": dept_name, "index": i + 1,
                    "url": url, "error": str(e), "reviews": []
                })

            human_sleep(2, 4)

    # ── 5. Save ───────────────────────────────────────────────────────────────
    with open("amazon_bestsellers.json", "w", encoding="utf-8") as f:
        json.dump(all_products, f, ensure_ascii=False, indent=2)

    df = pd.DataFrame(all_reviews_flat)
    df.to_csv("amazon_bestsellers_reviews.csv", index=False)
    print(f"\n💾 Saved: amazon_bestsellers.json + amazon_bestsellers_reviews.csv ({len(df)} reviews)")

    summary = pd.DataFrame([{
        "department":      p.get("department"),
        "index":           p.get("index"),
        "name":            p.get("name", "N/A")[:45],
        "price":           p.get("price", "N/A"),
        "rating_score":    p.get("rating_score", "N/A"),
        "rating_count":    p.get("rating_count", "N/A"),
        "reviews_scraped": len(p.get("reviews", [])),
    } for p in all_products])

    print(f"\n📊 Summary ({len(all_products)} products across {len(DEPARTMENTS)} departments):")
    display(summary)
    display(df.head(10))

except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()
    print(f"\n🔍 URL  : {driver.current_url}")
    print(f"🔍 Title: {driver.title}")

finally:
    time.sleep(2)
    driver.quit()
    print("🛑 Driver closed")